# CEQ v17-K -- Kaggle reproduction notebook

Round v17-K. This notebook does not decide anything: **the certified local
RTX 4060 DECIDES every research number in this round; this notebook only
REPRODUCES on Kaggle's GPU and prints `|Delta|` against the local reading.**
A number that exists only because this notebook produced it is not a
research result -- every cell below labels its own output REPRODUCTION and
names the local cell it is checked against.

**Nothing secret lives in this notebook.** No token, no key, no private
path is written into any cell. A committed Kaggle notebook is a published
artifact from the moment it is pushed; treat it that way while editing it,
not only before publishing it. If a push-to-hub step is ever added, its
credentials come from an interactive `huggingface_hub.login()` prompt or the
`HF_TOKEN` environment variable, never from a literal.

**Kaggle facts this notebook is built against** (`TRAINING.md` section 3,
repo-recorded 2026-08-25 -- re-check the session cap and quota before
trusting a multi-week schedule): 12 h per session, 30 GPU-h/week published
quota, `/kaggle/working` is 20 GB and auto-saved across sessions, GPU is
P100-16GB or 2xT4-16GB (this notebook uses one GPU; no multi-GPU code path
exists in `ceq/hf/train.py` and none is added here -- NO NEW
CONSTRUCTIONS), **internet is OFF inside the notebook** by default.

## Gate order -- each section refuses to proceed if the one before it did not

1. **Environment** -- device, torch version, free VRAM.
2. **L1** -- repo cloned at a pinned SHA, SHA asserted; pinned packages
   checked/installed; a CPU-only package smoke test.
3. **L1** -- datasets attached, SHA-256 of every dataset printed and
   asserted against `results/k_data_manifest.json`.
4. **L2 -- K-CERT.** `scripts/k_cert.py`'s device certificate. If the worst
   `delta/tol` clears 50%, this notebook HALTS and no cell below counts.
5. **L3 -- the queue.** Q1 reproduces the local deciding reading (Q2 was
   dropped by RULING 8 -- see the tombstone cell in the queue below), Q3 is
   training in <=11h chunks with bitwise resume across sessions, Q4
   reproduces the capability table.

Full cell-by-cell notes, the dependency list, and the launch checklist are
in `V17_NOTEBOOK.md` at the repo root -- **read it before pushing this
notebook.** The short version of the checklist:

- [ ] fill in `PINNED_SHA` (and `REPO_URL` or `CODE_DATASET_DIR`) in the
  clone cell
- [ ] attach the public datasets and upload the private ones via Kaggle's
  "Add Data" (exact slugs and licences in `V17_NOTEBOOK.md`) -- BED-M is
  **not** one of the private uploads; all three synthetic beds regenerate
  from a seed
- [ ] run `python scripts/k_cert.py` once locally and commit
  `results/k_cert_local.json` -- the module exists and its own tests pass,
  but nobody has produced the certificate file yet (see `V17_NOTEBOOK.md`
  "Dependencies")
- [ ] confirm `ceq/kdata.py`'s three Kaggle-side pins are filled in (they
  ship `UNPINNED_AWAITING_KAGGLE`) by running the L1 hash cell once, pasting
  the printed digests into `results/k_data_manifest.json`, and committing
- [ ] pick GPU: Settings > Accelerator > GPU T4 x2 (or P100)
- [ ] decide the clone path: a code-snapshot Dataset (no network) or
  Internet ON for the clone/install cells only

In [ ]:
# 1. ENVIRONMENT. No network calls in this cell.
import os, sys, time, platform

# CUBLAS_WORKSPACE_CONFIG BEFORE TORCH TOUCHES THE DEVICE, NOT LATER.
# `use_deterministic_algorithms(True)` needs it for cuBLAS reductions, and set
# after CUDA has initialised it makes strict mode fail the FORWARD too
# (measured, V17_R4_RETAKE_PRICE.md). The symptom is ORDER DEPENDENCE, not an
# error: measured locally, `tests/gate0` reads 1 failed / 188 passed without it
# and 189 passed with it, the failure being a Gate-0 identity bind that passes
# in isolation either way. A notebook whose cell order differs from the local
# suite would produce a bind failure that LOOKS like a refutation and is not.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

def gate(name, ok, msg=""):
    """Refuse to proceed: every later section ends by calling this. A failed
    gate raises, which halts a Kaggle "Run All" at the cell that actually
    broke, instead of letting a later cell produce a number downstream of a
    precondition that silently did not hold."""
    print("[{}] {}".format("OK" if ok else "HALT", name) + (" -- " + msg if msg else ""))
    if not ok:
        raise RuntimeError("GATE FAILED: {}: {}".format(name, msg))

print("python ", sys.version)
try:
    import torch
except ImportError:
    torch = None
gate("torch importable", torch is not None,
     "" if torch is not None else "the Kaggle base image should ship torch already")

print("torch  ", torch.__version__)
gate("CUBLAS_WORKSPACE_CONFIG set before torch",
     os.environ.get("CUBLAS_WORKSPACE_CONFIG") == ":4096:8",
     "got {!r} -- if Kaggle imported torch before this cell ran, the run "
     "header must record which determinism regime actually applied"
     .format(os.environ.get("CUBLAS_WORKSPACE_CONFIG")))
cuda = torch.cuda.is_available()
print("cuda available", cuda)
if cuda:
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print("  device {}: {}  free {:.2f} GiB / total {:.2f} GiB".format(
            i, torch.cuda.get_device_name(i), free / 1024**3, total / 1024**3))
else:
    print("  NO GPU VISIBLE -- Settings > Accelerator > GPU, then Run All again")

gate("a CUDA device is visible", cuda, "select a GPU accelerator in Notebook Settings")

## L1 -- repo at a pinned SHA, pinned packages, a CPU-only smoke test

In [ ]:
# 2. L1 -- repo cloned at a pinned SHA, SHA asserted.
#    Internet is OFF by default on Kaggle. Two paths, tried in order:
#      (a) a code-snapshot Dataset attached under /kaggle/input, already at
#          the pinned SHA -- no network needed;
#      (b) `git clone` + `checkout`, which needs the notebook's Internet
#          toggle turned ON for this cell only.
#    Either way the SHA is asserted below; a wrong SHA halts regardless of
#    which path produced the checkout. A resumed session (this directory
#    already exists from an earlier chunk) skips both and re-checks the SHA.
import shutil, subprocess

PINNED_SHA = ""          # <-- AUTHOR FILLS THIS IN BEFORE PUSHING. Required.
REPO_URL = ""             # <-- e.g. 'https://github.com/<you>/<repo>.git'
CODE_DATASET_DIR = ""     # <-- e.g. '/kaggle/input/ceq-repo-snapshot', if attached
REPO_DIR = "/kaggle/working/ceq-project"

gate("PINNED_SHA is set", bool(PINNED_SHA), "fill in PINNED_SHA above")

def run(cmd, **kw):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True, text=True, capture_output=True, **kw)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("{} already present (resumed session) -- not re-cloning".format(REPO_DIR))
elif CODE_DATASET_DIR and os.path.isdir(CODE_DATASET_DIR):
    print("copying code snapshot from {} (no network)".format(CODE_DATASET_DIR))
    shutil.copytree(CODE_DATASET_DIR, REPO_DIR)
elif REPO_URL:
    print("cloning from network -- requires Internet ON in Notebook Settings")
    run(["git", "clone", "-q", REPO_URL, REPO_DIR])
    run(["git", "checkout", "-q", PINNED_SHA], cwd=REPO_DIR)
else:
    gate("a code source is configured", False, "set CODE_DATASET_DIR or REPO_URL above")

got_sha = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
print("HEAD is", got_sha)
gate("HEAD == PINNED_SHA", got_sha == PINNED_SHA,
     "got {}, expected {}".format(got_sha, PINNED_SHA))

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

In [ ]:
# 3. Pin versions. Kaggle's base image ships torch/transformers/datasets
#    already; this only installs what is missing or wrong-versioned, so a
#    clean run needs no network at all when the image already matches.
#    requirements.txt pins transformers==5.3.0 tight -- TRAINING.md section
#    6.1: modeling_ceq.py sets six undocumented PreTrainedModel internals
#    that a later transformers release (including a patch) can rename.
#    python-chess is added here for ceq/kdata.py's label recomputation
#    (chess games source); it is not in requirements.txt today.
import importlib

NEED = {"transformers": "5.3.0", "datasets": "4.8.4",
        "huggingface_hub": "1.7.1", "accelerate": None, "chess": None}
missing = []
for mod, want in NEED.items():
    try:
        m = importlib.import_module(mod)
        have = getattr(m, "__version__", None)
        flag = "  (want {})".format(want) if want and have != want else ""
        print("  {:<16} {}{}".format(mod, have, flag))
        if want and have != want:
            missing.append("{}=={}".format(mod, want))
    except ImportError:
        print("  {:<16} NOT INSTALLED".format(mod))
        missing.append("{}=={}".format(mod, want) if want else mod)

if missing:
    print("installing (requires Internet ON):", missing)
    !pip -q install -U {" ".join(missing)}
else:
    print("all pinned packages already present -- no install needed")

In [ ]:
# 3b. Cheap CPU-only package smoke test before spending any GPU-hour. Seven
#     checks, no network, no GPU (`ceq/hf/smoke.py`); catches a broken
#     environment before K-CERT or the queue wastes quota on it.
!cd {REPO_DIR} && python -m ceq.hf.smoke
gate("ceq.hf.smoke passes (7 checks, CPU only)", _exit_code == 0,
     "exit code {}".format(_exit_code))

## L1 -- dataset attach + hash pin

In [ ]:
# 4. L1 -- the author's dataset attach list, AUTHORITATIVE. Do not add or
#    substitute a source here -- NO NEW CONSTRUCTIONS extends to data
#    sources too.
#
#    Public sets: attach via Kaggle's "Add Data" before Run All.
#    Private sets: the author uploads them himself as private Datasets.
#
#    DO NOT ATTACH -- struck by the author on licence/provenance grounds:
#      robikscube/this-week-in-chess-archive
#      dimitrioskourtikakis/gm-games-chesscom
#      thedevastator/tinystories-narrative-classification
#      nightfury1103/enwik8
#      lanceni/enwik8            (100,000,000 B -- right size, licence unknown)
#      nguyenatu/enwik8          (apache-2.0 but BPE tokenizer JSONs, not the corpus)
#      yorkyong/text8-zip        (unknown licence, a different cut)
DATASET_PATHS = {
    # arevel/chess-games -- Lichess PGN, CC0, 1.56 GB. python-chess
    # recomputes legality and next-FEN on the fly; no stored label is read.
    "lichess_chess_games": "/kaggle/input/chess-games",
    # lichess/chess-evaluations -- Lichess + Stockfish evals, CC0, 34.4 GB.
    # ONE shard is streamed and hashed below; the set is never pulled whole.
    "lichess_chess_evaluations": "/kaggle/input/chess-evaluations",
    # enwik8 -- Kaggle attach jamesmcguigan/hutter-prize (CC-BY-SA-3.0),
    # member enwik9. enwik8 IS the first 100,000,000 bytes of enwik9 -- the
    # Hutter Prize's own definition, and MEASURED here: that slice hashes to
    # the pin already in the manifest. No upload, no unlicensed mirror.
    "enwik8": "/kaggle/input/hutter-prize",
    # alexkarev/tinystories-train-ready -- CDLA-Sharing-1.0, 595 MB.
    "tinystories_cdla": "/kaggle/input/tinystories-train-ready",
    # BED-M, BED-K, BED-1 -- ALL THREE regenerated from generator+seed;
    # nothing to attach. CORRECTED from an earlier draft of this notebook,
    # which treated BED-M as an intact upload: the attach list's "211,765
    # line" description belongs to data/tinystories_20k.txt, not to a chain
    # corpus, and BED-M has no on-disk artifact at all -- it is
    # `ceq.corpus.build()` (named as BED-M by `ceq/beds/__init__.py:3`).
    # `results/k_data_manifest.json` pins it as `kind: generator` at
    # `ceq.corpus.build(n_train=384, n_test=128, seed=0)`, exactly like
    # BED-K / BED-1 below.
}

for name, path in DATASET_PATHS.items():
    tag = "FOUND" if os.path.isdir(path) else "NOT ATTACHED YET"
    print("  {:<24} {}  {}".format(name, path, tag))

In [ ]:
import hashlib
# 5. L1 -- print and assert the SHA-256 of every dataset at load, against
#    `results/k_data_manifest.json`. NOTHING BELOW REIMPLEMENTS HASHING OR
#    SPLITTING -- `ceq/kdata.py` owns it (implemented; 29/29 of its own
#    tests green as of this notebook's authoring -- see V17_NOTEBOOK.md
#    "Dependencies" for how to re-check that before trusting this cell).
#
#    THE THREE KAGGLE-SIDE SOURCES START UNPINNED. `results/k_data_manifest.json`
#    ships with `lichess_chess_games`, `lichess_chess_evaluations` and
#    `tinystories_cdla` at `status: UNPINNED_AWAITING_KAGGLE` -- nobody has
#    run this cell against the real files yet. `kdata.verify_digest` PRINTS
#    the measured digest FIRST, then raises `kdata.MissingPin` because there
#    is nothing to compare it to (an absent pin is an error, never a silent
#    skip -- ceq/kdata.py's own rule). FIRST RUN: this cell halts here;
#    copy the printed digest into `results/k_data_manifest.json`'s
#    `sources.<name>.sha256`, flip `status` to `PINNED`, commit, and re-run
#    -- only then is the hash actually being ASSERTED rather than measured.
#
#    FILENAMES INSIDE EACH ATTACHED DATASET ARE PLACEHOLDERS below (a
#    Kaggle dataset's internal layout is not known until it is attached) --
#    fill them in from the actual attached directory listing.
from ceq import kdata

manifest = kdata.load_manifest()
print("manifest sources:", sorted(manifest["sources"]))

# arevel/chess-games -- intact file, hashed and asserted whole.
games_pgn = os.path.join(DATASET_PATHS["lichess_chess_games"], "<pinned-filename>.pgn")
kdata.verify_file("lichess_chess_games", games_pgn, manifest=manifest)

# lichess/chess-evaluations -- ONE streamed shard, hashed while streaming,
# never slurped whole (the set is 34.4 GB; the shard is not).
evals_path = os.path.join(DATASET_PATHS["lichess_chess_evaluations"], "<pinned-shard-filename>")
with open(evals_path, "rb") as fh:
    reader = kdata.HashedLineReader(fh)
    for _ in reader:
        pass
    kdata.verify_digest("lichess_chess_evaluations", reader.hexdigest(), manifest=manifest)

# enwik8 -- attached as a SLICE of a licensed superset: the first
# 100,000,000 bytes of hutter-prize's `enwik9`. `read_enwik8_slice` raises
# `kdata.ShortRead` rather than hashing a truncated mount -- a partial read
# that still produces a digest is worse than one that fails.
# Already pinned (author-measured): sha256 2b49720ec4d78c3c9fabaee6e4179a5e997302b3a70029f30f2d582218c024a8
# over the extracted 100,000,000-byte file, split offsets 90,042,869 / 95,000,818.
enwik9_path = os.path.join(DATASET_PATHS["enwik8"], "enwik9")
with open(enwik9_path, "rb") as _fh:
    enwik8_bytes = kdata.read_enwik8_slice(_fh)
kdata.verify_digest("enwik8", hashlib.sha256(enwik8_bytes).hexdigest(),
                    manifest=manifest)

# alexkarev/tinystories-train-ready
tinystories_path = os.path.join(DATASET_PATHS["tinystories_cdla"], "<pinned-filename>")
kdata.verify_file("tinystories_cdla", tinystories_path, manifest=manifest)

# BED-M, BED-K, BED-1 -- ALL THREE regenerated from generator+seed; assert
# the SIGNATURE rather than a file hash (there is no file to hash, BED-M
# included -- see the dataset-attach cell above for why).
for name in ("bed_m", "bed_k", "bed_1"):
    sig = kdata.bed_signature(name)
    print("  {} regenerated, signature {}".format(name, sig))
    assert sig == manifest["sources"][name]["sha256"], (
        "{}: regenerated signature does not match the pinned manifest".format(name))

gate("every dataset hash matches the pinned manifest", True)

In [ ]:
# 5b. L1 -- REAL-DATA SMOKE TEST for ceq/kdata.py's hygiene guards, before
#     K-CERT and before anything counts. The five hygiene rules are proven
#     in tests/gate0/test_g05_data.py against 36 synthetic PGN games sharing
#     three opening trunks and a 24-article synthetic enwik8-shaped fixture
#     -- neither has ever seen a real Lichess game (a variant, a truncated
#     mainline, an odd header) or the real enwik8 <page> stream. This
#     notebook is the only thing that ever touches the real 1.56 GB / 100 MB
#     files, so that gap closes HERE or nowhere. Anything the guards cannot
#     handle raises and HALTs this cell -- consistent with every other L1
#     gate in this notebook.
import itertools, pathlib

# ---- chess: the first ~100 REAL games through the real loader ----
real_games = list(itertools.islice(kdata.iter_games(games_pgn), 100))
gate("at least 100 real games parsed from the attached PGN",
     len(real_games) >= 100, "got {}".format(len(real_games)))

real_keys = [k for k, _ in real_games]
real_assign = kdata.split_by_game(real_keys)

n_plies = 0
for _, game in real_games:
    # label_plies() replays the REAL mainline through python-chess -- a
    # variant, a non-standard header, or a truncated game surfaces here as
    # an exception, not as a silently-empty result.
    n_plies += len(kdata.label_plies(game))
gate("every real game replayed through python-chess without raising",
     n_plies > 0, "{} games, {} plies total".format(len(real_games), n_plies))

kdata.check_game_split([(k, real_assign[k]) for k in real_keys])
counts = {s: sum(1 for v in real_assign.values() if v == s) for s in ("train", "val", "test")}
print("real-data smoke (chess): {} games, {} plies, split {}".format(
    len(real_games), n_plies, counts))

# ---- enwik8: the first few REAL <page> boundaries ----
# `enwik8_bytes` is built in the verify cell above, straight from the
# guarded slice -- re-reading it here would read a file that no longer exists.
real_starts = kdata.article_starts(enwik8_bytes)
gate("at least 3 real <page> boundaries found in the attached enwik8",
     len(real_starts) >= 3, "got {}".format(len(real_starts)))
real_bounds = kdata.split_by_article(enwik8_bytes)   # raises HygieneViolation on its own if a snap fails
kdata.check_article_bounds(enwik8_bytes, real_bounds)
print("real-data smoke (enwik8): {} real <page> boundaries, split bounds {}".format(
    len(real_starts), {k: v for k, v in real_bounds.items() if k in ("train", "val", "test")}))

gate("real-data smoke test passed for both chess and enwik8", True)

## L2 -- K-CERT

Writes a device certificate. **If its worst `delta/tol` clears 50%, this
notebook HALTS here and no cell below counts** -- the contract's own rule,
restated as code rather than as a comment.

In [ ]:
# 6. L2 -- K-CERT. scripts/k_cert.py does not reimplement anything it
#    certifies against; it is called, not rewritten. As authored, the
#    module itself is implemented (its pure functions pass their own unit
#    tests -- tests/gate0/test_g06_kcert.py -k "not certificate"), but IT
#    HAS NOT BEEN RUN ONCE LOCALLY YET: `results/k_cert_local.json` does
#    not exist in this tree, so the gate below halts until that lands.
#    tests/gate0/test_g06_kcert.py is the source of truth for the interface
#    (fit_power_law, delta_over_tol, halt_on_bar, HALT_FRACTION,
#    zero_step_gate, determinism_verdict, and the k_cert_local.json schema)
#    -- see V17_NOTEBOOK.md "Dependencies" for how to re-check both facts.
import json, pathlib

try:
    from scripts import k_cert
    HALT_FRACTION = k_cert.HALT_FRACTION
except ImportError:
    HALT_FRACTION = 0.50   # the contract's own literal; scripts/k_cert.py not importable yet

LOCAL_REFERENCE_CERT = pathlib.Path(REPO_DIR) / "results" / "k_cert_local.json"
gate("the local certified reading is present in this clone",
     LOCAL_REFERENCE_CERT.exists(),
     "{} missing -- it must be committed at PINNED_SHA".format(LOCAL_REFERENCE_CERT))

# k_cert.py's output path is not configurable by CLI flag as documented
# today, so move the local reference aside before re-running it here, or the
# Kaggle run overwrites the file this comparison needs.
local_ref_saved = LOCAL_REFERENCE_CERT.with_name("k_cert_local_REFERENCE.json")
shutil.copy(LOCAL_REFERENCE_CERT, local_ref_saved)

!cd {REPO_DIR} && python -u scripts/k_cert.py 2>&1 | tee -a /kaggle/working/train.log

kaggle_cert_path = pathlib.Path("/kaggle/working/k_cert_kaggle.json")
shutil.move(str(LOCAL_REFERENCE_CERT), str(kaggle_cert_path))   # the run above just wrote here

local_cert = json.loads(local_ref_saved.read_text())
kaggle_cert = json.loads(kaggle_cert_path.read_text())

# The certificate schema is under active development elsewhere in this repo
# (tests/gate0/test_g06_kcert.py is the source of truth). Check the shape
# before indexing into it, so a stale/partial k_cert_local.json fails with a
# clear message here instead of a bare TypeError three lines down.
for tag, cert in (("local", local_cert), ("kaggle", kaggle_cert)):
    gate("{} certificate has a bar.worst reading".format(tag),
         isinstance(cert.get("bar", {}).get("worst"), dict),
         "{} is missing bar.worst -- re-run scripts/k_cert.py; "
         "`python -m pytest tests/gate0/test_g06_kcert.py -q` names the exact gap".format(
             kaggle_cert_path if tag == "kaggle" else LOCAL_REFERENCE_CERT))

print("local  box:", local_cert["box"], " git:", local_cert["git"]["head"])
print("kaggle box:", kaggle_cert["box"], " git:", kaggle_cert["git"]["head"])
print("local  worst delta/tol:", local_cert["bar"]["worst"]["delta_over_tol"],
      " halt:", local_cert["bar"]["halt"])
print("kaggle worst delta/tol:", kaggle_cert["bar"]["worst"]["delta_over_tol"],
      " halt:", kaggle_cert["bar"]["halt"])

gate("Kaggle K-CERT did not HALT", kaggle_cert["bar"]["halt"] is None,
     str(kaggle_cert["bar"]["halt"]))
gate("Kaggle worst delta/tol clears the 50% line",
     kaggle_cert["bar"]["worst"]["delta_over_tol"] < HALT_FRACTION,
     str(kaggle_cert["bar"]["worst"]))

print("K-CERT GREEN on this device. The queue below may run.")

## L3 -- the queue

### Q1 -- reproduction of the local deciding reading

**Never a verdict.** Every number below is printed beside its local
counterpart with a `|Delta|` column.

**INFERRED INSTRUMENT, NOT NAMED VERBATIM IN THE v17-K TASK SPEC:**
`scripts/v15_r1.py --device cuda` is the only in-tree, CUDA-capable runner
that already produces a certified-device reading in this repo (recent
commits: "Certify CUDA, refute the 5.8x gap", "Price the deciding cells on
the certified device") and its `--tag` writes `results/{tag}.jsonl` in the
exact record shape this cell reads. **Confirm this is v17-K's actual Q1
before relying on it** -- swap the command and the field list below if the
round names a different deciding measurement. See `V17_NOTEBOOK.md` "What
the author must confirm".

In [ ]:
# 7. Q1 -- reproduction, not a verdict. See the markdown note above for
#    why `scripts/v15_r1.py` is the instrument used here. (Q2 dropped,
#    RULING 8 -- see the tombstone cell below.)
# RULING 4's re-take. `results/v15_r1.jsonl` is the SUPERSEDED cpu journal --
# it keeps its records under L-G2 and carries an append-only supersede marker
# as line 26; it is history, not the comparison set. The certified reading is
# 24 cells on the 4060 under warn_only=True, each carrying instrument_hash
# 5d41a63d...9a309 (RULING 5).
LOCAL_Q1_JSONL = pathlib.Path(REPO_DIR) / "results" / "v17k_r4_retake.jsonl"   # <-- CONFIRM this is the CERTIFIED CUDA reading, not a stale cpu-tagged file
KAGGLE_TAG = "v17k_kaggle_q1"

gate("a local reference reading is present", LOCAL_Q1_JSONL.exists(), str(LOCAL_Q1_JSONL))

# All THREE arms: `arm_smprime` IS R1-prime and is the workhorse, so a
# reproduction omitting it is not the cross-device table the round asked for.
!cd {REPO_DIR} && python -u scripts/v15_r1.py --device cuda --arms arm_pl arm_smprime softmax --seeds 0 1 2 3 4 5 6 7 --tag {KAGGLE_TAG} 2>&1 | tee -a /kaggle/working/train.log

def _load_jsonl(p):
    return [json.loads(line) for line in pathlib.Path(p).read_text().splitlines() if line.strip()]

local_rows = _load_jsonl(LOCAL_Q1_JSONL)
kaggle_rows = _load_jsonl(pathlib.Path(REPO_DIR) / "results" / (KAGGLE_TAG + ".jsonl"))

def _cells(rows):
    return {(r["kind"], r["seed"]): r for r in rows if r.get("t") == "cell"}

local_cells, kaggle_cells = _cells(local_rows), _cells(kaggle_rows)
# scripts/v15_r1.py's t="cell" record fields, read from the file directly
# rather than assumed -- confirm against the version at PINNED_SHA.
NUMERIC_FIELDS = ("eval_nrmse", "eval_h_hat", "conservation_drift", "a_hat_max")

print("{:<16}{:<6}{:<24}{:>14}{:>14}{:>14}".format(
    "kind", "seed", "field", "local", "kaggle", "|Delta|"))
for key in sorted(set(local_cells) & set(kaggle_cells)):
    for field in NUMERIC_FIELDS:
        lv, kv = local_cells[key].get(field), kaggle_cells[key].get(field)
        if isinstance(lv, (int, float)) and isinstance(kv, (int, float)):
            print("{:<16}{:<6}{:<24}{:>14.6f}{:>14.6f}{:>14.6e}  REPRODUCTION, not a verdict"
                  .format(key[0], key[1], field, lv, kv, abs(lv - kv)))
missing_keys = set(local_cells) ^ set(kaggle_cells)
if missing_keys:
    print("WARNING: (kind, seed) present on one side only:", sorted(missing_keys))

### Q2 -- DROPPED (RULING 8)

**There is no cell here to run.** Q2 (R2 reproduction, `t*=8`, `n=16,384`)
is dropped, not skipped. The gap in the queue between Q1 above and Q3 below
is deliberate and is the honest record of what happened -- not a
renumbering. This repo's L-G2 principle: superseded and dropped things are
marked, never erased.

**Why.** `arm_smprime` at `n=16,384` reserves **10.578 GiB** under a
training loop against the certified RTX 4060's **7.996 GiB** total
`[MEASURED, COSTS.md section 1.6 / results/k_cert_local.json]`, so the shape does
not fit the certified device. The same shape fits a T4 (59% of its
budget), which places Q2's deciding cell exactly where this round's KILL
clause strikes it: **local decides, and local cannot run it.** There was
no re-take to price because there was nothing to re-take. Ruling filed
verbatim in `V17K_RULINGS.md`, RULING 8.

**What this does not settle.** RULING 8 states plainly that dropping the
*reproduction* does not supply the *reading* -- whether the round carries
an R2 row at all, given R2's deciding cell cannot run on the certified
device either, is flagged there as a separate, open question, not
answered by this tombstone or by silence.

Q1, Q3, Q4 below keep their names; nothing is renumbered.

### Q3 -- training, chunked <=11h, bitwise resume across sessions

`out_dir` is a NEW directory every chunk; `resume_from` points at the
previous one (`ceq/hf/train.py`'s own convention, `TRAINING.md` section
6.6). This cell auto-detects the newest checkpoint already sitting in the
auto-saved `/kaggle/working` and resumes from it, so re-running the
notebook in a fresh Kaggle session picks up where the last one stopped.

In [ ]:
# 8. Q3 -- calibrate s/step on THIS session's GPU before committing to a
#    chunk size. TRAINING.md section 4: "Do not compute this from a table.
#    Measure it in the first session." Two T4 extrapolations already in
#    this repo disagree by 4-9x, so neither is trusted here.
from ceq.hf import train as T
import inspect

# CORPUS SOURCE: the L1 hash-verified enwik8 TRAIN split, not
# `T.load_open_text(...)` (TRAINING.md section 6.4's pattern). That call is
# `datasets.load_dataset(..., streaming=True)`, which fetches shards over
# the network at iteration time -- confirmed a hard blocker under this
# contract's internet-off rule (V17_G04_ENV.md "B4": "with internet OFF
# this cannot work at all"). enwik8's train bytes were already verified in
# L1 (`enwik8_bytes`, `kdata.split_by_article`); reusing them here needs no
# second read and no network.
CORPUS_PATH = "/kaggle/working/corpus.txt"
if not os.path.exists(CORPUS_PATH):
    lo, hi = real_bounds["train"]
    pathlib.Path(CORPUS_PATH).write_bytes(enwik8_bytes[lo:hi])
print("{:.2f} MiB corpus (enwik8 train split, byte-verified in L1)".format(
    os.path.getsize(CORPUS_PATH) / 1024**2))

CONFIG = dict(T.DEFAULTS)      # hidden 512, L8, H8, seq 512, batch 8
# T4-16GB, not P100: ceq/sizing.GPUS has no P100 entry, and T4 is the more
# conservative of the two Kaggle offers (TRAINING.md section 6.3).
ok, msg = T.preflight(gpu="T4-16GB", grad_checkpoint=False, **CONFIG)
print(msg)
gate("the shape fits the certified budget", ok, msg)

CALIB_OUT = "/kaggle/working/ceq-run-calib"
t0 = time.time()
_ = T.train(out_dir=CALIB_OUT, steps=50, device="cuda", data_path=CORPUS_PATH,
           gpu="T4-16GB", grad_checkpoint=False, log_every=10, **CONFIG)
r_secs_per_step = (time.time() - t0) / 50
print("measured r = {:.4f} s/step on this session's GPU".format(r_secs_per_step))

CHUNK_CAP_H = 11.0    # the task's own ceiling; strictly under the 12h session cap
CHUNK_STEPS = int(0.80 * CHUNK_CAP_H * 3600 / r_secs_per_step)
print("chunk_steps = floor(0.80 * {}h / {:.4f} s/step) = {:,}".format(
    CHUNK_CAP_H, r_secs_per_step, CHUNK_STEPS))

In [ ]:
# 9. Q3 -- ONE CHUNK of training this session, resuming bitwise from the
#    last checkpoint if `/kaggle/working` carries one.
#
#    G0.3 K-PERSIST (7/7 green): the checkpoint WRITE is now atomic --
#    `ceq/hf/train.py::_atomic_torch_save` writes a sibling `.tmp`, fsyncs,
#    then `os.replace`s it, so a kill between truncate and last byte cannot
#    leave a present-but-unloadable `trainer_state.pt` (measured failure
#    mode before the fix: a 57,636-of-115,272-byte file that looks
#    resumable and is not). `train()` also now REFUSES `resume_from ==
#    out_dir`, so the notebook's own new-directory-per-chunk convention
#    below is required, not merely recommended.
#
#    REMAINING GAP, STILL OPEN (see V17_NOTEBOOK.md "Dependencies"):
#    the checkpoint is written ONCE, at the very end of the `steps` loop --
#    atomically, but only once. A session killed mid-loop, before `steps`
#    completes, still loses the WHOLE chunk's progress, not just its tail;
#    there is no periodic mid-chunk checkpoint. This cell is written against
#    the interface it NEEDS (`T.train(..., save_every=N)`, atomically
#    checkpointing inside the loop) and uses it if present; otherwise it
#    warns loudly and keeps going with today's checkpoint-only-at-the-end
#    behaviour, so CHUNK_STEPS should be kept well under the 11h line until
#    this lands, not run right up against it.
import contextlib, glob, re

class _Tee:
    """Duplicate stdout to /kaggle/working/train.log so a killed session
    leaves a log (python -u + tee, the same rule TRAINING.md applies to the
    subprocess cells, applied here to an in-process call)."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, s):
        for st in self.streams:
            st.write(s)
    def flush(self):
        for st in self.streams:
            st.flush()

RUN_PREFIX = "/kaggle/working/ceq-run-"
existing = sorted(glob.glob(RUN_PREFIX + "[0-9][0-9][0-9]"))
prev = existing[-1] if existing else None
next_idx = (int(re.search(r"(\d+)$", prev).group(1)) + 1) if prev else 0
out_dir = "{}{:03d}".format(RUN_PREFIX, next_idx)

print("previous checkpoint:", prev or "(none -- fresh run)")
print("this chunk writes to:", out_dir)

train_kwargs = dict(out_dir=out_dir, steps=CHUNK_STEPS, device="cuda",
                    data_path=CORPUS_PATH, gpu="T4-16GB", grad_checkpoint=False,
                    log_every=50, resume_from=prev, **CONFIG)
if "save_every" in inspect.signature(T.train).parameters:
    train_kwargs["save_every"] = max(1, CHUNK_STEPS // 20)   # ~20 atomic checkpoints per chunk
else:
    print("WARNING: ceq.hf.train.train() has no save_every parameter yet -- "
          "this chunk is NOT safe against a mid-chunk kill. See V17_NOTEBOOK.md.")

with open("/kaggle/working/train.log", "a", encoding="utf-8") as logf, \
     contextlib.redirect_stdout(_Tee(sys.stdout, logf)):
    r = T.train(**train_kwargs)

print("params", "{:,}".format(r["n_params"]), " peak GiB", r["peak_bytes"] / 1024**3)
print("start_step", r["start_step"], "-> now", r["start_step"] + r["steps"])
print("final loss", r["losses"][-1], " worst row L1", min(r["row_l1_min"]))

if prev:
    !python -u -c "import torch; d=torch.load('{prev}/trainer_state.pt', map_location='cpu', weights_only=True); print(sorted(d.keys())); print('resumed from step', d['step'])"

# --- RULING 10' -- "PINNED" BY LIKELIHOOD RATIO. The criterion, evaluated.
#
#       Lambda = 2 * [ LL_eval(beta_final) - LL_eval(beta == 1) ]
#
#     from TWO DETERMINISTIC FORWARD PASSES on the HELD-OUT eval split. No
#     training, no identical-seed pair, no floor. Verdict:
#
#       Lambda <= 3.841   chi2_1 at 0.95 (Wilks)  ->  PINNED
#       Lambda >  ln n    BIC, n the eval count   ->  MOVED
#       between them  ->  "rejected at 0.95, below description-length"
#
#     BOTH constants print on every line, and so does the resolution
#     |beta-1|_min = sqrt(3.841 / (n * I_beta)). "PINNED" means
#     INDISTINGUISHABLE AT THIS RESOLUTION, never an implication of exactness,
#     so a verdict printed without its minimum detectable departure is a defect
#     (RULING 10'). `lrt_report` is what prints them together, so the defect is
#     unreachable from here rather than merely discouraged.
#
#     THE SPLIT IS HELD OUT BY CONSTRUCTION, NOT BY ARITHMETIC DONE HERE.
#     `T.ByteBatches` cuts the corpus 90/10 contiguously and `train()` draws
#     ONLY from `.train`; this cell draws from `.val`, off the same object built
#     from the same bytes. Nothing here re-derives an offset that could drift
#     from the one training used.
#
#     `5*delta_beta` (RULING 2a) is RETIRED AS THE CRITERION and kept as a
#     DIAGNOSTIC. It is printed underneath for exactly one purpose: a diagnostic
#     that CONTRADICTS the verdict is a finding, and it cannot contradict
#     anything if it stops being computed. `scripts/k_noise_floor.py` keeps
#     measuring delta_beta and that is correct.
#
#     THE WALD CROSS-CHECK IS NOT A FREE CONFIRMATION. `(beta-1)^2 * I_beta`
#     against the same 3.841 is asymptotically the same statistic, but the
#     equivalence holds AT THE MLE and `beta_final` is fit on the TRAINING
#     split, so it never is one. Measured locally at a 27,914-parameter shape:
#     Lambda = 7.21 (rejects) against Wald = 1.15 (does not), OPPOSITE sides of
#     3.841 on the same beta (`V17_R10P_LRT.md`). Where the two split,
#     `lrt_report` prints DISAGREES; read both, and do not report one alone.
import torch as _t
from ceq.hf.modeling_ceq import CEQForCausalLM, beta_lrt, beta_summary, lrt_report

#: The SAME 90/10 cut `train()` made, off the same bytes, so ".val" is text the
#: chunk above never saw. `max_bytes` is read off `train`'s own signature rather
#: than retyped, because a different cap here would be a different corpus.
_max_bytes = inspect.signature(T.train).parameters["max_bytes"].default
_data = T.ByteBatches(T._corpus_text(CORPUS_PATH, _max_bytes),
                      vocab_size=CONFIG["vocab_size"])
#: Fixed generator seed: the eval batch must be the SAME batch every chunk, or
#: Lambda is not comparable across the chunks it is printed in.
_eval_x, _ = _data.batch("val", CONFIG["batch"], CONFIG["seq"],
                         _t.Generator().manual_seed(10_1993), r["device"])
#: `labels=_eval_x`, not the shifted tensor: the model shifts internally
#: (`logits[:, :-1]` against `labels[:, 1:]`), the same call `train()` makes.
_model = CEQForCausalLM.from_pretrained(out_dir).to(r["device"]).eval()

_before = {n: p.detach().clone() for n, p in _model.named_parameters()}
_lrt = beta_lrt(_model, input_ids=_eval_x, labels=_eval_x)
print(lrt_report(_lrt))

#: THE RESTORE PROOF, printed rather than trusted. The second pass substitutes
#: beta and must leave the model bit-identical: torch.equal, never allclose.
_intact = all(_t.equal(_before[n], p) for n, p in _model.named_parameters())
print("restore proof -- torch.equal over all {} parameters: {}".format(
    len(_before), _intact))
assert _intact, "beta_lrt did not restore the model bitwise"

#: THE RETIRED DIAGNOSTIC, beside the verdict it no longer decides. `branch`
#: is None until the floor node lands delta_beta, and the VERDICT above does
#: not wait on it -- that independence is the whole point of RULING 10'.
print("delta_beta DIAGNOSTIC (RULING 2a, retired as the criterion by 10'):")
print(" ", beta_summary(r["beta"], census=r.get("beta_census")))

with open("/kaggle/working/lrt_verdict.json", "w") as _fh:
    json.dump(_lrt, _fh, indent=2)


### Q4 -- capability table reproduction

`scale/m3_quintuple.py`'s own docstring: cuda rows are "tolerance-checked
against their CPU counterparts, never bitwise". The LOCAL cuda journal
(`results/m3_quintuple_v2_cuda.jsonl`, already committed at `PINNED_SHA`) is
the certified reading this cell reproduces against -- it is what
`m3_quintuple.py --device cuda` is CERTIFIED to have produced on the RTX
4060, not a fresh invention of this notebook.

In [ ]:
# 10. Q4 -- capability table REPRODUCTION.
#     m3_quintuple.py's cuda journal path is hardcoded
#     (results/m3_quintuple_v2_cuda.jsonl, results/m3_quintuple_v2_cuda_weights/)
#     with no --tag / output-path override, so the local reference is moved
#     aside before this cell reruns the cuda lane on Kaggle's GPU.
LOCAL_Q4_JOURNAL = pathlib.Path(REPO_DIR) / "results" / "m3_quintuple_v2_cuda.jsonl"
gate("the local Q4 journal is present", LOCAL_Q4_JOURNAL.exists(), str(LOCAL_Q4_JOURNAL))
local_q4_saved = LOCAL_Q4_JOURNAL.with_name("m3_quintuple_v2_cuda_REFERENCE.jsonl")
shutil.copy(LOCAL_Q4_JOURNAL, local_q4_saved)

# --task / --seeds / --steps default to the registered reading
# (scale/m3_quintuple.py's own SHIPPED_TASK and defaults) -- not overridden
# here, so this reproduces the same cell the local journal was priced on.
!cd {REPO_DIR} && python -u scale/m3_quintuple.py --device cuda 2>&1 | tee -a /kaggle/working/train.log

kaggle_q4_journal = LOCAL_Q4_JOURNAL   # the run above just overwrote it with the Kaggle-side rows
local_q4_rows = _load_jsonl(local_q4_saved)
kaggle_q4_rows = _load_jsonl(kaggle_q4_journal)
print("local reference rows: {}   kaggle rows: {}".format(len(local_q4_rows), len(kaggle_q4_rows)))

!cd {REPO_DIR} && python -m scale.capability_table --journal {kaggle_q4_journal} --version kaggle_v17k 2>&1 | tee -a /kaggle/working/train.log
print("wrote results/capability_table_kaggle_v17k.{md,json} -- REPRODUCTION; "
      "diff its numbers against results/capability_table_v0.md / v1.md by hand, "
      "never cite the Kaggle table as a verdict")

## What this notebook does not do

- It does not decide anything. Every Q1/Q4 number is a reproduction with
  a `|Delta|` column against a local, certified reading; a Kaggle-only
  number is not a research result (see the top of this notebook).
- It does not push anything to the Hugging Face Hub. If that step is added
  later, follow `TRAINING.md` section 6.8: `huggingface_hub.login()`
  interactive or `HF_TOKEN` from the environment, never a literal in a
  cell -- this notebook is a published artifact once it is on Kaggle.
- It does not reimplement hashing, dataset splitting, device certification,
  or training/checkpointing -- it calls `ceq/kdata.py`, `scripts/k_cert.py`,
  and `ceq/hf/train.py` as shipped. See `V17_NOTEBOOK.md` for exactly what
  each of those still needs before this notebook can run end to end.